In [1]:
%pip install scikit-learn

  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached narwhals-2.28.0-py3-none-any.whl.metadata (15 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 27.7 MB/s  0:00:00 23.2 MB/s eta 0:00:01
Using cached joblib-1.6.0-py3-none-any.whl (306 kB)
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)
Using cached narwhals-2.28.0-py3-none-any.whl (490 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 40.7 MB/s  0:00:00 42.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [scikit-learn]0m 5/6 [scikit-learn]]
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

variables = pd.read_parquet("data/clean/variables_modelo.parquet")

X = variables.drop(columns=["id_cliente", "sexo", "impago_mes_siguiente"])
y = variables["impago_mes_siguiente"]
print("Variables del modelo:", X.shape[1])

Variables del modelo: 18


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)
print("Entrenamiento:", len(X_train), "· Prueba:", len(X_test))
print("Impago en cada parte:", round(y_train.mean(), 3), round(y_test.mean(), 3))

Entrenamiento: 20936 · Prueba: 8973
Impago en cada parte: 0.221 0.221


In [5]:
modelo = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000),
)
modelo.fit(X_train, y_train)

pd_test = modelo.predict_proba(X_test)[:, 1]
print("Primeras PD:", pd_test[:5].round(3))

Primeras PD: [0.177 0.366 0.14  0.131 0.103]


In [6]:
auc = roc_auc_score(y_test, pd_test)
gini = 2 * auc - 1
print(f"AUC: {auc:.3f} · Gini: {gini:.3f}")

AUC: 0.739 · Gini: 0.478


In [7]:
logistica = modelo.named_steps["logisticregression"]
pesos = pd.DataFrame({
    "variable": X.columns,
    "peso": logistica.coef_[0],
}).sort_values("peso", key=abs, ascending=False)
pesos["efecto"] = np.where(pesos["peso"] > 0, "sube el riesgo", "baja el riesgo")
display(pesos.round(3))

,variable,peso,efecto
0,retraso_sep,0.370,sube el riesgo
1,meses_con_retraso,0.370,sube el riesgo
2,peor_retraso_6m,0.312,sube el riesgo
4,limite_credito,-0.212,baja el riesgo
7,parte_pagada_6m,0.166,sube el riesgo
11,nivel_estudios_desconocido,-0.118,baja el riesgo
15,estado_civil_desconocido,-0.089,baja el riesgo
17,estado_civil_soltero,-0.078,baja el riesgo
8,cambio_uso,0.069,sube el riesgo
6,uso_medio_6m,0.062,sube el riesgo


In [8]:
cliente = X_test.iloc[[0]]
pd_cliente = modelo.predict_proba(cliente)[0, 1]

escalado = modelo.named_steps["standardscaler"].transform(cliente)[0]
aporte = pd.Series(escalado * logistica.coef_[0], index=X.columns).sort_values(key=abs, ascending=False)

print(f"PD de este cliente: {pd_cliente:.1%}")
display(aporte.head(5).round(2).to_frame("aporte al riesgo"))

PD de este cliente: 17.7%


,aporte al riesgo
meses_con_retraso,-0.20
peor_retraso_6m,-0.20
limite_credito,0.19
cambio_uso,0.14
parte_pagada_6m,-0.09


In [9]:
import joblib
joblib.dump(modelo, "data/clean/modelo_logistico.joblib")

['data/clean/modelo_logistico.joblib']